In [0]:
import builtins
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, MapType, DoubleType
from datetime import datetime, timedelta
import random
# Generate sample transactions for multiple months, ensuring multiple transactions per month
def generate_transactions(num_transactions):
    transactions = []
    months = [1, 2, 3, 4]  # Jan-Apr
    for i in range(num_transactions):
        month = random.choice(months)
        day = random.randint(1, 28)
        transaction = {
            "transaction_id": f"T{i+1}",
            "transaction_date": f"2026-{month:02d}-{day:02d}",
            "transaction_amount": builtins.round(random.uniform(50, 500), 2)
        }
        transactions.append(transaction)
    return transactions
# Generate sample transactions for multiple months
def generate_transactions(num_transactions):
    transactions = []
    base_date = datetime(2026, 1, 1)
    for i in range(num_transactions):
        transaction = {
            "transaction_id": f"T{i+1}",
            "transaction_date": (base_date + timedelta(days=random.randint(0, 120))).strftime("%Y-%m-%d"),
            "transaction_amount": builtins.round(random.uniform(50, 500), 2)
        }
        transactions.append(transaction)
    return transactions

customer_data = [
    {
        "customer_name": "Alice",
        "age": 34,
        "transactions": generate_transactions(7)
    },
    {
        "customer_name": "Bob",
        "age": 28,
        "transactions": generate_transactions(6)
    },
    {
        "customer_name": "Charlie",
        "age": 45,
        "transactions": generate_transactions(7)
    },
    {
        "customer_name": "Diana",
        "age": 39,
        "transactions": generate_transactions(6)
    },
    {
        "customer_name": "Ethan",
        "age": 52,
        "transactions": generate_transactions(7)
    }
]

schema = StructType([
    StructField("customer_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("transactions", ArrayType(
        StructType([
            StructField("transaction_id", StringType(), True),
            StructField("transaction_date", StringType(), True),
            StructField("transaction_amount", DoubleType(), True)
        ])
    ), True)
])

df = spark.createDataFrame(customer_data, schema)
display(df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_transactions = df.withColumn('transactions',explode('transactions'))
display(df_transactions)

In [0]:
df_newColumns = df_transactions.withColumn('transaction_date',col("transactions.transaction_date"))\
                                .withColumn('Amount',col("transactions.transaction_amount"))\
                                .drop('transactions')\
                                .withColumn('transaction_month',date_format('transaction_date','yyyy-MM'))                                    
df_newColumns.show()

df_result = df_newColumns.groupBy('customer_name','age','transaction_month').agg(sum("Amount"))
display(df_result)